# Escalation Dataset & Audit Preparation Pipeline for Gating Neural Network
## Dual Parallel LLM Calls (v1 With Label & v2 No Label) + SpanNER 160 (10-Class) + Longest-Span NMS

This notebook produces comprehensive training, evaluation, and audit datasets for the **Gating Neural Network (Gating NN)**:
- **Batched & Multi-Threaded Execution ($B = 5$ abstracts / batch, 10 concurrent LLM calls)**:
  1. **SpanNER 160 Model**: Inferences 10 canonical classes (`best_spanner_160train.pt`).
  2. **Non-Maximum Suppression (NMS)**: Longest-Span First policy with highest-confidence tie-breaker.
  3. **Dual Dynamic Prompt Construction**:
     - **Prompt v1 (With Label)**: `predicted_label`, `second_best_label`, `uncertainty`, `novelty_score`, `margin`, `p_background_o`.
     - **Prompt v2 (No Label)**: `uncertainty`, `novelty_score`, `margin`, `p_background_o` (no predicted categories).
  4. **Parallel LLM Dispatch ($2 \times B = 10$ calls/batch)**: Dispatches v1 and v2 simultaneously across all batch abstracts.
  5. **Zero Label Arbitration**: Stores pure SpanNER and pure LLM predictions for Gating NN training.
  6. **Per-Abstract Storage in `Gating NN/`**:
     - `Gating NN/json/<doc_id>.json`: Complete structured data, ground truth, SpanNER features, v1/v2 outputs, raw prompts & LLM responses.
     - `Gating NN/reports/<doc_id>.txt`: Full side-by-side audit report, evidence breakdown, matrix, and abstract-level metrics.
  7. **Global Aggregated Datasets**:
     - `Gating NN/gating_dataset_400train_dual_prompts.csv` and `.jsonl`
     - `Gating NN/gating_dataset_400train_v1.csv` and `gating_dataset_400train_v2.csv`


In [ ]:
# ==============================================================================
# [CELL 1]: Environment Setup & Module Imports
# ==============================================================================
import os
import sys
import glob
import json
import time
import math
import warnings
from datetime import datetime
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from dotenv import load_dotenv

# Suppress warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Set Project & Workspace Roots
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
WORKSPACE_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, '..'))

for p in [PROJECT_ROOT, WORKSPACE_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load environment variables
load_dotenv(os.path.join(PROJECT_ROOT, '.env'), override=True)

# Internal LinkNER modules
import importlib
from src.common.label_mapping import (
    CANONICAL_10_LABELS,
    CANONICAL_10_LABEL2ID,
    CANONICAL_10_ID2LABEL,
    to_new_label,
    canonicalize_label
)

from src.architectures.hybrid_linkner.spanner_model import SpanNERModel
from src.data.dataset_converter import parse_ann_file
from src.architectures.hybrid_linkner.pipeline import get_default_llm_provider
from src.architectures.hybrid_linkner.novelty_detector import build_lof_novelty_detector, LOFNoveltyDetector

# Single-call prompt utilities
from src.architectures.gating_nn.prompt_templates import (
    load_prompt_template,
    format_inlined_abstract,
    format_entities_evidence_v1,
    format_entities_evidence_v2,
    build_single_call_prompt,
    parse_single_call_llm_response
)

print('✅ All modules imported successfully!')
print(f'  • Project Root  : {PROJECT_ROOT}')
print(f'  • Workspace Root: {WORKSPACE_ROOT}')
print(f'  • 10 Canonical Classes: {CANONICAL_10_LABELS}')


In [ ]:
# ==============================================================================
# [CELL 2]: Experiment Configuration & Output Paths in 'Gating NN'
# ==============================================================================
# 1. Dataset Directory (400Abstracts no_disease train split)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/new/400Abstracts/no_disease/train')
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(PROJECT_ROOT, 'data/new/400Abstracts/train')

# 2. Model & Checkpoint Paths
TOKEN_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models/NER_Model/trained_NER_model')
SPANNER_CHECKPOINT = os.path.join(PROJECT_ROOT, 'models/SpanNER_LSF/best_spanner_160train.pt')
if not os.path.exists(SPANNER_CHECKPOINT):
    SPANNER_CHECKPOINT = os.path.join(PROJECT_ROOT, 'models/SpanNER_LSF/best_spanner_160train.pt')

# 3. Batching & Concurrency Parameters
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '5'))                # Abstracts per batch (B = 5)
MAX_WORKERS = min(int(os.getenv('MAX_WORKERS', '10')), 16)   # 2 * B = 10 parallel LLM workers per batch
UNCERTAINTY_METHOD = os.getenv('UNCERTAINTY_METHOD', 'mcd').lower()  # 'mcd', 'pe', or 'lc'
MCD_PASSES = 5
MAX_SPAN_WIDTH = 6
MAX_ABSTRACTS = None                                          # Set to None for all 240, or e.g. 5 for a test batch

# 4. Output Directories inside 'Gating NN'
GATING_NN_DIR = os.path.join(WORKSPACE_ROOT, 'Gating NN')
GATING_JSON_DIR = os.path.join(GATING_NN_DIR, 'json')
GATING_REPORTS_DIR = os.path.join(GATING_NN_DIR, 'reports')

os.makedirs(GATING_NN_DIR, exist_ok=True)
os.makedirs(GATING_JSON_DIR, exist_ok=True)
os.makedirs(GATING_REPORTS_DIR, exist_ok=True)

print('=' * 88)
print('  📋 DUAL PROMPT ESCALATION EXPERIMENT CONFIGURATION')
print('=' * 88)
print(f'  • Data Directory        : {DATA_DIR}')
print(f'  • SpanNER Checkpoint    : {SPANNER_CHECKPOINT}')
print(f'  • Batch Size (B)        : {BATCH_SIZE} abstracts / batch')
print(f'  • Parallel LLM Workers  : 2 × {BATCH_SIZE} = {2 * BATCH_SIZE} concurrent threads (v1 & v2)')
print(f'  • NMS Policy            : Longest Span First (Tie-breaker: Highest Probability)')
print(f'  • Input Features        : predicted_label, second_best_label, uncertainty, novelty_score, margin, p_background_o')
print(f'  • Output Directory      : {GATING_NN_DIR}')
print(f'    ├── Per-Abstract JSON : {GATING_JSON_DIR}')
print(f'    ├── Audit Reports TXT : {GATING_REPORTS_DIR}')
print(f'    └── Aggregated Tables : {GATING_NN_DIR}/gating_dataset_400train_*.csv')
print('=' * 88)


In [ ]:
# ==============================================================================
# [CELL 3]: Load Dataset Abstracts and Ground Truth Annotations
# ==============================================================================
all_ann_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.ann')))
selected_files = []

for ann_path in all_ann_files:
    txt_path = ann_path.replace('.ann', '.txt')
    if os.path.exists(txt_path):
        gt_ents = parse_ann_file(ann_path)
        clean_gt = []
        for g in gt_ents:
            c_label = canonicalize_label(g['label'], use_new_labels=True)
            if c_label not in {'O', 'Lifestyle_factor', 'lifestyle_factor'}:
                clean_gt.append({
                    'id': g.get('id', ''),
                    'text': g['text'],
                    'start_char': g['start_char'],
                    'end_char': g['end_char'],
                    'label': c_label
                })
        selected_files.append((txt_path, ann_path, clean_gt))

if MAX_ABSTRACTS is not None:
    selected_files = selected_files[:MAX_ABSTRACTS]

total_gt_count = sum(len(gt) for _, _, gt in selected_files)
print(f'✅ Successfully loaded {len(selected_files)} paired abstracts from: {DATA_DIR}')
print(f'   Total Ground Truth Entities: {total_gt_count}')


In [ ]:
# ==============================================================================
# [CELL 4]: Initialize & Fit LOF Vector Novelty Detector on SpanNER 160 Training Distribution
# ==============================================================================
# NOTE: SpanNER 160 was fine-tuned on the 160 abstracts from 200Abstracts (120 Train + 40 Val).
# To measure true novelty relative to SpanNER's training distribution (Zero Leakage),
# the reference vector database is strictly constructed from those 160 fine-tuning abstracts.

print('🔍 Initializing LOF Novelty Detector on SpanNER 160 Fine-Tuning Corpus (160 Abstracts)...')
from src.architectures.hybrid_linkner.novelty_detector import get_default_embedder, LOFNoveltyDetector, build_lof_novelty_detector

# 1. Initialize Active Embedder (Qwen3-Embedding-8B configured in .env)
active_embedder = get_default_embedder()
print(f'  • Active Embedder: {active_embedder}')

# 2. Load the 160 Fine-Tuning Abstracts (120 Train + 40 Val from 200Abstracts)
SPANNER_TRAIN_160_JSON = os.path.join(PROJECT_ROOT, 'data/processed/spanner_dataset_160_10class.json')
train_160_exemplars = []
seen_160 = set()

if os.path.exists(SPANNER_TRAIN_160_JSON):
    with open(SPANNER_TRAIN_160_JSON, 'r', encoding='utf-8') as f:
        ds_160 = json.load(f)
    for doc in ds_160.get('train', []):
        doc_id = doc.get('doc_id', 'doc')
        for s in doc.get('spans', []):
            t = s.get('text', '').strip()
            lbl = s.get('label', '')
            if t and lbl not in ('O', 'Lifestyle_factor', 'lifestyle_factor') and t.lower() not in seen_160:
                seen_160.add(t.lower())
                train_160_exemplars.append({
                    'text': t,
                    'label': lbl,
                    'abstract_id': doc_id,
                    'source': 'spanner_160_train'
                })
else:
    # Fallback to loading .ann files from 200Abstracts/train and 200Abstracts/val
    f_160 = glob.glob(os.path.join(PROJECT_ROOT, 'data/new/200Abstracts/train/*.ann')) + \
            glob.glob(os.path.join(PROJECT_ROOT, 'data/new/200Abstracts/val/*.ann'))
    for ann_f in f_160:
        doc_id = os.path.basename(ann_f).replace('.ann', '')
        for g in parse_ann_file(ann_f):
            c_label = canonicalize_label(g['label'], use_new_labels=True)
            t = g['text'].strip()
            if t and c_label not in ('O', 'Lifestyle_factor', 'lifestyle_factor') and t.lower() not in seen_160:
                seen_160.add(t.lower())
                train_160_exemplars.append({
                    'text': t,
                    'label': c_label,
                    'abstract_id': doc_id,
                    'source': 'spanner_160_train'
                })

# 3. Embed all 160-abstract training exemplars with Qwen 8B and fit LOF Detector
print(f'  • Embedding {len(train_160_exemplars)} unique training exemplars from SpanNER 160 fine-tuning corpus...')
texts_to_embed = [e['text'] for e in train_160_exemplars]
embeddings = active_embedder.embed(texts_to_embed)
vector_db = []
for emb, rec in zip(embeddings, train_160_exemplars):
    rec_copy = rec.copy()
    rec_copy['embedding'] = emb
    vector_db.append(rec_copy)

novelty_detector = LOFNoveltyDetector(vector_db=vector_db, k=10, alpha=0.5, embedder=active_embedder)
print(f'✅ LOF Novelty Detector successfully fitted on {len(novelty_detector.matrix)} exemplars from SpanNER 160 training distribution (Embedding Dim: {novelty_detector.matrix.shape[1]}).')


In [ ]:
# ==============================================================================
# [CELL 5]: Load 10-Class SpanNER Model (160-Abstract Checkpoint)
# ==============================================================================
device = torch.device('mps') if torch.backends.mps.is_available() else (
    torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
)

print(f'⚡ Loading SpanNER 10-class model on {device}...')
spanner_model = SpanNERModel(encoder_path=TOKEN_MODEL_DIR, num_classes=10, max_span_width=MAX_SPAN_WIDTH)
spanner_model.load_state_dict(torch.load(SPANNER_CHECKPOINT, map_location=device))
spanner_model.to(device)
spanner_model.eval()

tokenizer = AutoTokenizer.from_pretrained(TOKEN_MODEL_DIR)
print(f'✅ SpanNER 160 model loaded successfully from {SPANNER_CHECKPOINT}!')


In [ ]:
# ==============================================================================
# [CELL 6]: SpanNER 160 Prediction & Longest-Span NMS Feature Extractor
# ==============================================================================
def run_nms(spans):
    """Non-Maximum Suppression: Resolves overlapping candidate spans by Longest Span First (tie-breaker: highest prob)."""
    sorted_spans = sorted(spans, key=lambda x: (x['end_char'] - x['start_char'], x.get('prob', 0.0)), reverse=True)
    kept = []
    for s in sorted_spans:
        overlap = False
        for k in kept:
            if not (s['end_char'] <= k['start_char'] or s['start_char'] >= k['end_char']):
                overlap = True
                break
        if not overlap:
            kept.append(s)
    return sorted(kept, key=lambda x: x['start_char'])

def match_gt_label(span_text, start_char, end_char, gt_ents):
    """Matches candidate span against ground truth annotations (exact offset, exact text, or overlap)."""
    p_norm = span_text.strip().lower()
    for g in gt_ents:
        g_norm = g['text'].strip().lower()
        exact_char = (start_char == g['start_char'] and end_char == g['end_char'])
        exact_text = (p_norm == g_norm)
        overlap = not (end_char <= g['start_char'] or start_char >= g['end_char'])
        if exact_char or exact_text or (overlap and (p_norm in g_norm or g_norm in p_norm)):
            return g['label']
    return 'O'

def extract_spanner_abstract_spans(text, gt_ents, uncertainty_method='mcd', mcd_passes=5):
    """
    Runs SpanNER 160 inference on an abstract, computes Top-1, Top-2, Margin, Uncertainty,
    p_background_o (P(O)), filters via Longest-Span NMS, computes LOF Novelty scores, and matches GT labels.
    """
    sp_enc = tokenizer([text], padding=True, truncation=True, max_length=512, return_offsets_mapping=True, return_tensors='pt')
    input_ids = sp_enc['input_ids'].to(device)
    attention_mask = sp_enc['attention_mask'].to(device)
    offsets = sp_enc['offset_mapping'][0].cpu().numpy()

    with torch.no_grad():
        out = spanner_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out['logits'][0]  # [Num_Spans, 10]
        candidate_spans = out['candidate_spans']
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)

        top1_probs = sorted_probs[:, 0].cpu().numpy()
        top1_indices = sorted_indices[:, 0].cpu().numpy()
        top2_probs = sorted_probs[:, 1].cpu().numpy()
        top2_indices = sorted_indices[:, 1].cpu().numpy()
        margins = top1_probs - top2_probs
        p_background_o = probs[:, 0].cpu().numpy()  # Index 0 is background 'O'

    # Uncertainty Computation
    method = uncertainty_method.lower()
    if method == 'mcd':
        spanner_model.train()  # Enable dropout
        mcd_probs_list = []
        with torch.no_grad():
            for _ in range(mcd_passes):
                out_m = spanner_model(input_ids=input_ids, attention_mask=attention_mask)
                mcd_probs_list.append(F.softmax(out_m['logits'][0], dim=-1))
        spanner_model.eval()
        mcd_stack = torch.stack(mcd_probs_list, dim=0)  # [M, Num_Spans, 10]
        mcd_entropies = -torch.sum(mcd_stack * torch.log(mcd_stack + 1e-9), dim=-1)
        avg_mcd_entropy = torch.mean(mcd_entropies, dim=0).cpu().numpy()
        uncertainties = avg_mcd_entropy / math.log(10)
    elif method == 'pe':
        raw_entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1).cpu().numpy()
        uncertainties = raw_entropy / math.log(10)
    else:  # LC (Least Confidence)
        uncertainties = 1.0 - top1_probs

    raw_spans = []
    for s_idx, (start, end, _) in enumerate(candidate_spans):
        lbl_id = int(top1_indices[s_idx])
        if lbl_id == 0:  # Skip background 'O'
            continue

        char_start = int(offsets[start, 0])
        char_end = int(offsets[end, 1])
        span_text = text[char_start:char_end].strip()
        if not span_text:
            continue

        pred_label = CANONICAL_10_ID2LABEL[lbl_id]
        second_label = CANONICAL_10_ID2LABEL[int(top2_indices[s_idx])]

        raw_spans.append({
            'span_text': span_text,
            'start_char': char_start,
            'end_char': char_end,
            'predicted_label': pred_label,
            'spanner_label': pred_label,
            'second_best_label': second_label,
            'prob': float(top1_probs[s_idx]),
            'top2_prob': float(top2_probs[s_idx]),
            'margin': float(margins[s_idx]),
            'uncertainty': float(uncertainties[s_idx]),
            'u_score': float(uncertainties[s_idx]),
            'p_background_o': float(p_background_o[s_idx]),
            'prob_o': float(p_background_o[s_idx]),
        })

    # Apply Longest-Span NMS
    nms_spans = run_nms(raw_spans)

    # Compute LOF Novelty
    if nms_spans and novelty_detector is not None:
        texts_to_embed = [s['span_text'] for s in nms_spans]
        vecs = novelty_detector.embed(texts_to_embed)
        nov_scores = novelty_detector.novelty(vecs)
        for s, nov in zip(nms_spans, nov_scores):
            s['novelty_score'] = float(nov)
            s['novelty'] = float(nov)
    else:
        for s in nms_spans:
            s['novelty_score'] = 0.0
            s['novelty'] = 0.0

    # Match Ground Truth Label
    for s in nms_spans:
        s['gt_label'] = match_gt_label(s['span_text'], s['start_char'], s['end_char'], gt_ents)

    return nms_spans

print('✅ SpanNER 160 + Longest-Span NMS Feature Extractor ready.')


In [ ]:
def classify_outcome(gt_label: str, spanner_label: str, llm_label: str) -> str:
    g = gt_label
    s = spanner_label
    f = llm_label

    is_g_pos = (g != 'O' and g != 'Non_LSF' and g != 'LSF_out_of_context')
    is_s_pos = (s != 'O' and s != 'Non_LSF' and s != 'LSF_out_of_context')
    is_f_pos = (f != 'O' and f != 'Non_LSF' and f != 'LSF_out_of_context')

    if is_g_pos:
        if s != g and f == g:
            return 'FIXED_ENTITY (GT_X_SPAN_Y_LLM_X)'
        elif s == g and f == g:
            return 'PRESERVED_CORRECT (GT_X_SPAN_X_LLM_X)'
        elif s == g and f != g:
            return 'RUINED (GT_X_SPAN_X_LLM_Y)'
        elif s != g and is_f_pos and f != g:
            return 'CONFUSED_CATEGORY (GT_X_SPAN_Y_LLM_Z)'
        elif s != g and not is_f_pos:
            return 'DROPPED_ENTITY (GT_X_SPAN_X_LLM_O)'
    else:
        if is_s_pos and not is_f_pos:
            return 'FILTERED_FP (GT_O_SPAN_X_LLM_O)'
        elif is_s_pos and is_f_pos:
            return 'RETAINED_FP (GT_O_SPAN_X_LLM_X)'
        else:
            return 'CONFIRMED_NEGATIVE (GT_O_SPAN_O_LLM_O)'
    return 'OTHER'

# ==============================================================================
# [CELL 7]: Per-Abstract Report & JSON Serialization Engines
# ==============================================================================
def generate_abstract_audit_report(doc_id, text, gt_ents, spans, prompt_v1_str, resp_v1_str, prompt_v2_str, resp_v2_str, records):
    """
    Generates a structured, human-readable text audit report (.txt) for an abstract,
    logging raw prompts, raw LLM completions, candidate evidence, and comparative audit matrices.
    """
    lines = []
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    lines.append('=' * 100)
    lines.append(f'📄 ABSTRACT AUDIT & ESCALATION REPORT: [DOC_ID: {doc_id}]')
    lines.append(f'Run Timestamp: {timestamp} | Dataset Split: {os.path.basename(DATA_DIR)}')
    lines.append(f'Model Checkpoint: SpanNER 160 (10 Canonical Classes) | NMS Policy: Longest Span First')
    lines.append(f'LLM Architecture: Dual Parallel Consolidated Prompts (Prompt v1 With Label & Prompt v2 No Label)')
    lines.append('=' * 100)
    lines.append('')
    
    # 1. Raw Abstract & Ground Truth
    lines.append('-' * 100)
    lines.append('1. RAW ABSTRACT TEXT & GROUND TRUTH ENTITIES')
    lines.append('-' * 100)
    lines.append('Text:')
    lines.append(f'"{text}"')
    lines.append('')
    lines.append(f'Ground Truth Entities ({len(gt_ents)} entities):')
    if gt_ents:
        for idx, g in enumerate(gt_ents, 1):
            lines.append(f'  • [T{idx}] ({g["start_char"]}, {g["end_char"]}): "{g["text"]}" -> {g["label"]}')
    else:
        lines.append('  • (No positive ground truth entities)')
    lines.append('')
    
    # 2. SpanNER 160 Predictions
    lines.append('-' * 100)
    lines.append(f'2. SPANNER 160 INFERENCE & NMS CANDIDATE SPANS (Total Candidates: {len(spans)})')
    lines.append('-' * 100)
    for idx, s in enumerate(spans, 1):
        lines.append(f'[Span {idx}]: \'{s["span_text"]}\' (Chars: {s["start_char"]}-{s["end_char"]})')
        lines.append(f'  • Top-1 Class   : {s["predicted_label"]} (Prob: {s["prob"]:.4f})')
        lines.append(f'  • Top-2 Class   : {s["second_best_label"]} (Prob: {s["top2_prob"]:.4f}) | Margin: {s["margin"]:.4f} | P(O): {s["p_background_o"]:.4f}')
        lines.append(f'  • Uncertainty   : {s["uncertainty"]:.4f} ({UNCERTAINTY_METHOD.upper()}) | Novelty: {s["novelty_score"]:.4f} (LOF)')
        lines.append(f'  • Ground Truth  : {s["gt_label"]} | Status: {"✅ CORRECT" if s["spanner_label"] == s["gt_label"] and s["gt_label"] != "O" else "❌ WRONG / EXTRA"}')
        lines.append('')
    
    # 3. Prompt v1 Input & Raw Response
    lines.append('-' * 100)
    lines.append('3. PROMPT V1 (WITH LABEL) - INPUT PROMPT & RAW LLM RESPONSE')
    lines.append('-' * 100)
    lines.append('--- [PROMPT V1 INPUT STRING] ---')
    lines.append(prompt_v1_str)
    lines.append('')
    lines.append('--- [PROMPT V1 RAW LLM RESPONSE] ---')
    lines.append(resp_v1_str if resp_v1_str.strip() else '(Empty Response)')
    lines.append('')
    
    # 4. Prompt v2 Input & Raw Response
    lines.append('-' * 100)
    lines.append('4. PROMPT V2 (NO LABEL) - INPUT PROMPT & RAW LLM RESPONSE')
    lines.append('-' * 100)
    lines.append('--- [PROMPT V2 INPUT STRING] ---')
    lines.append(prompt_v2_str)
    lines.append('')
    lines.append('--- [PROMPT V2 RAW LLM RESPONSE] ---')
    lines.append(resp_v2_str if resp_v2_str.strip() else '(Empty Response)')
    lines.append('')
    
    # 5. Side-by-Side Comparison Matrix
    lines.append('-' * 100)
    lines.append('5. SIDE-BY-SIDE ENTITY COMPARISON & AUDIT MATRIX')
    lines.append('-' * 100)
    lines.append(f'{"Span Text":<28} | {"Chars":<11} | {"Ground Truth":<18} | {"SpanNER 160":<18} | {"Prompt v1 (With)":<18} | {"Prompt v2 (No)":<18} | {"Concordance":<12}')
    lines.append('-' * 135)
    for r in records:
        stext = (r['span_text'][:25] + '...') if len(r['span_text']) > 28 else r['span_text']
        ch_str = f"{r['start_char']}-{r['end_char']}"
        agree_str = "AGREE" if r['v1_vs_v2_agree'] else "DISAGREE"
        lines.append(f"{stext:<28} | {ch_str:<11} | {r['gt_label']:<18} | {r['spanner_label']:<18} | {r['llm_v1_label']:<18} | {r['llm_v2_label']:<18} | {agree_str:<12}")
    lines.append('')
    
    # 6. Abstract-Level Performance Summary
    lines.append('-' * 100)
    lines.append('6. ABSTRACT-LEVEL PERFORMANCE METRICS')
    lines.append('-' * 100)
    n_cand = len(records)
    if n_cand > 0:
        sp_c = sum(r['spanner_correct'] for r in records)
        v1_c = sum(r['llm_v1_correct'] for r in records)
        v2_c = sum(r['llm_v2_correct'] for r in records)
        or_c = sum(r['oracle_correct'] for r in records)
        ag_c = sum(r['v1_vs_v2_agree'] for r in records)
        
        lines.append(f'  • Total Evaluated Candidates: {n_cand}')
        lines.append(f'  • SpanNER 160 Accuracy      : {sp_c}/{n_cand} ({sp_c/n_cand*100:.1f}%)')
        lines.append(f'  • Prompt v1 Accuracy (With) : {v1_c}/{n_cand} ({v1_c/n_cand*100:.1f}%)')
        lines.append(f'  • Prompt v2 Accuracy (No)   : {v2_c}/{n_cand} ({v2_c/n_cand*100:.1f}%)')
        lines.append(f'  • Oracle Upper Bound        : {or_c}/{n_cand} ({or_c/n_cand*100:.1f}%)')
        lines.append(f'  • Prompt v1 vs v2 Agreement : {ag_c}/{n_cand} ({ag_c/n_cand*100:.1f}%)')
    else:
        lines.append('  • No candidate spans evaluated.')
    lines.append('=' * 100)
    
    return '\n'.join(lines)

def save_abstract_json(doc_id, text, gt_ents, spans, prompt_v1_str, resp_v1_str, prompt_v2_str, resp_v2_str, records, out_path):
    """
    Saves structured per-abstract output JSON including all candidate features, raw inputs/outputs, and metrics.
    """
    sp_c = sum(r['spanner_correct'] for r in records) if records else 0
    v1_c = sum(r['llm_v1_correct'] for r in records) if records else 0
    v2_c = sum(r['llm_v2_correct'] for r in records) if records else 0
    or_c = sum(r['oracle_correct'] for r in records) if records else 0
    
    doc_json = {
        'doc_id': doc_id,
        'dataset': DATA_DIR,
        'timestamp': datetime.now().isoformat(),
        'abstract_text': text,
        'ground_truth_entities': gt_ents,
        'candidate_spans': records,
        'raw_prompts_and_responses': {
            'prompt_v1_input': prompt_v1_str,
            'prompt_v1_response': resp_v1_str,
            'prompt_v2_input': prompt_v2_str,
            'prompt_v2_response': resp_v2_str
        },
        'abstract_metrics': {
            'total_candidates': len(records),
            'spanner_correct': sp_c,
            'llm_v1_correct': v1_c,
            'llm_v2_correct': v2_c,
            'oracle_correct': or_c
        }
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(doc_json, f, indent=2, ensure_ascii=False)

print('✅ Per-abstract report and JSON serializers ready.')


In [ ]:
# ==============================================================================
# [CELL 8]: LLM Provider & Consolidated Abstract Processor Engine
# ==============================================================================
llm_provider = get_default_llm_provider()
print(f'✅ Active LLM Provider: {type(llm_provider).__name__}')

def execute_single_llm_call(prompt):
    """Invokes LLM provider with error handling and returns raw response string."""
    try:
        return llm_provider.generate(prompt)
    except Exception as e:
        print(f'⚠️ [LLM Error]: {e}')
        return ''

def process_single_abstract(item):
    """
    Processes 1 abstract:
    1. Runs SpanNER 160 + Longest-Span NMS to extract candidate spans & neural features.
    2. Constructs Prompt v1 and Prompt v2.
    3. Executes both LLM calls in parallel.
    4. Parses outputs without arbitration.
    5. Generates and saves per-abstract report (.txt) and JSON (.json) inside 'Gating NN/'.
       (Always generates .json and .txt files for all 240 abstracts, even if 0 spans are detected).
    Returns list of candidate span records for global aggregation.
    """
    txt_path, ann_path, gt_ents = item
    doc_id = os.path.basename(txt_path).replace('.txt', '')
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # 1. SpanNER inference + Longest-Span NMS (with p_background_o)
    spans = extract_spanner_abstract_spans(text, gt_ents, uncertainty_method=UNCERTAINTY_METHOD, mcd_passes=MCD_PASSES)
    
    json_out_path = os.path.join(GATING_JSON_DIR, f'{doc_id}.json')
    report_out_path = os.path.join(GATING_REPORTS_DIR, f'{doc_id}.txt')
    
    # Handle abstracts where SpanNER detected 0 non-background candidate entities
    if not spans:
        prompt_v1_str = '(No candidate spans detected by SpanNER 160 to evaluate)'
        prompt_v2_str = '(No candidate spans detected by SpanNER 160 to evaluate)'
        resp_v1_str = '{"labeled_entities": []}'
        resp_v2_str = '{"labeled_entities": []}'
        records = []
        
        save_abstract_json(doc_id, text, gt_ents, spans, prompt_v1_str, resp_v1_str, prompt_v2_str, resp_v2_str, records, json_out_path)
        report_text = generate_abstract_audit_report(doc_id, text, gt_ents, spans, prompt_v1_str, resp_v1_str, prompt_v2_str, resp_v2_str, records)
        with open(report_out_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        return []

    # 2. Build Prompts
    prompt_v1_str = build_single_call_prompt(text, spans, version='v1')
    prompt_v2_str = build_single_call_prompt(text, spans, version='v2')

    # 3. Parallel Dual LLM Invocations
    with ThreadPoolExecutor(max_workers=2) as sub_pool:
        f_v1 = sub_pool.submit(execute_single_llm_call, prompt_v1_str)
        f_v2 = sub_pool.submit(execute_single_llm_call, prompt_v2_str)
        resp_v1_str = f_v1.result()
        resp_v2_str = f_v2.result()

    # 4. Parse Outputs
    parsed_v1 = parse_single_call_llm_response(resp_v1_str, spans)
    parsed_v2 = parse_single_call_llm_response(resp_v2_str, spans)

    # 5. Build Combined Records
    records = []
    for span, out_v1, out_v2 in zip(spans, parsed_v1, parsed_v2):
        gt_lab = span['gt_label']
        sp_lab = span['spanner_label']
        v1_lab = out_v1['llm_label']
        v2_lab = out_v2['llm_label']

        sp_corr = int(sp_lab == gt_lab and gt_lab != 'O')
        v1_corr = int(v1_lab == gt_lab and gt_lab != 'O')
        v2_corr = int(v2_lab == gt_lab and gt_lab != 'O')

        # v1 escalation benefit
        if v1_corr and not sp_corr:   v1_benefit = 1
        elif sp_corr and not v1_corr: v1_benefit = -1
        else:                         v1_benefit = 0

        # v2 escalation benefit
        if v2_corr and not sp_corr:   v2_benefit = 1
        elif sp_corr and not v2_corr: v2_benefit = -1
        else:                         v2_benefit = 0

        v1_out = classify_outcome(gt_lab, sp_lab, v1_lab)
        v2_out = classify_outcome(gt_lab, sp_lab, v2_lab)

        rec = {
            'doc_id': doc_id,
            'span_text': span['span_text'],
            'start_char': span['start_char'],
            'end_char': span['end_char'],
            'gt_label': gt_lab,
            # SpanNER 160 Features
            'spanner_label': sp_lab,
            'spanner_confidence': span['prob'],
            'spanner_top2_prob': span['top2_prob'],
            'second_best_label': span['second_best_label'],
            'margin': span['margin'],
            'uncertainty': span['uncertainty'],
            'uncertainty_method': UNCERTAINTY_METHOD.upper(),
            'novelty_score': span['novelty_score'],
            'p_background_o': span['p_background_o'],
            # Prompt v1 (With Label)
            'llm_v1_label': v1_lab,
            'llm_v1_predicted_tag': out_v1.get('llm_predicted_tag', ''),
            'llm_v1_rationale': out_v1.get('llm_rationale', ''),
            'llm_v1_correct': v1_corr,
            'v1_escalation_benefit': v1_benefit,
            'v1_outcome': v1_out,
            # Prompt v2 (No Label)
            'llm_v2_label': v2_lab,
            'llm_v2_rationale': out_v2.get('llm_rationale', ''),
            'llm_v2_correct': v2_corr,
            'v2_escalation_benefit': v2_benefit,
            'v2_outcome': v2_out,
            # Concordance & Correctness
            'spanner_correct': sp_corr,
            'v1_vs_v2_agree': int(v1_lab == v2_lab),
            'oracle_correct': int(sp_corr or v1_corr or v2_corr)
        }
        records.append(rec)

    # 6. Save Per-Abstract Report and JSON in 'Gating NN/'
    save_abstract_json(doc_id, text, gt_ents, spans, prompt_v1_str, resp_v1_str, prompt_v2_str, resp_v2_str, records, json_out_path)
    report_text = generate_abstract_audit_report(doc_id, text, gt_ents, spans, prompt_v1_str, resp_v1_str, prompt_v2_str, resp_v2_str, records)
    with open(report_out_path, 'w', encoding='utf-8') as f:
        f.write(report_text)

    return records


In [ ]:
# ==============================================================================
# [CELL 9]: Batched Multi-Threaded Execution Loop (B=5 abstracts, 10 LLM calls/batch)
# ==============================================================================
total_abstracts = len(selected_files)
total_batches = (total_abstracts + BATCH_SIZE - 1) // BATCH_SIZE

print('=' * 96)
print(f'  🚀 Launching Batched Pipeline: {total_abstracts} Abstracts in {total_batches} Batches (B={BATCH_SIZE})')
print(f'  • Concurrency: {BATCH_SIZE} SpanNER inferences + {2 * BATCH_SIZE} Parallel LLM Invocations per Batch')
print(f'  • Storage    : Per-Abstract JSON & Reports saved to: {GATING_NN_DIR}')
print('=' * 96)

all_records = []
t_start = time.time()

for b_idx in range(0, total_abstracts, BATCH_SIZE):
    batch_files = selected_files[b_idx : b_idx + BATCH_SIZE]
    batch_num = (b_idx // BATCH_SIZE) + 1
    current_bs = len(batch_files)
    doc_ids_in_batch = [os.path.basename(f[0]).replace('.txt', '') for f in batch_files]
    
    print(f'\n📦 [Batch {batch_num:02d}/{total_batches:02d}] Abstracts {b_idx + 1:3d}-{b_idx + current_bs:3d} of {total_abstracts} ({doc_ids_in_batch})...')
    t_b0 = time.time()
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as batch_executor:
        batch_results = list(batch_executor.map(process_single_abstract, batch_files))
        
    batch_records = [r for doc in batch_results for r in doc]
    all_records.extend(batch_records)
    
    t_b = time.time() - t_b0
    print(f'   -> Completed in {t_b:.1f}s | {len(batch_records)} candidate spans extracted across {current_bs} abstracts.')

total_elapsed = time.time() - t_start
print('=' * 96)
print(f'🏁 Pipeline Complete in {total_elapsed:.1f}s across {total_abstracts} abstracts!')
print(f'   Total Candidate Span Records: {len(all_records)}')
print('=' * 96)


In [ ]:
# ==============================================================================
# [CELL 10]: Global Comparative Analytics & Prompt Evaluation
# ==============================================================================
df_all = pd.DataFrame(all_records)
n_total = len(df_all)

print('=' * 96)
print('                 GLOBAL COMPARATIVE ESCALATION ANALYTICS')
print('=' * 96)

if n_total > 0:
    sp_corr = (df_all['spanner_correct'] == 1).sum()
    v1_corr = (df_all['llm_v1_correct'] == 1).sum()
    v2_corr = (df_all['llm_v2_correct'] == 1).sum()
    oracle_corr = (df_all['oracle_correct'] == 1).sum()

    v1_rescues = (df_all['v1_escalation_benefit'] == 1).sum()
    v1_degrades = (df_all['v1_escalation_benefit'] == -1).sum()
    v2_rescues = (df_all['v2_escalation_benefit'] == 1).sum()
    v2_degrades = (df_all['v2_escalation_benefit'] == -1).sum()
    concordance = (df_all['v1_vs_v2_agree'] == 1).sum()

    print(f'📊 Evaluated Candidate Spans : {n_total} spans across {len(selected_files)} abstracts')
    print(f'--------------------------------------------------------------------------------')
    print(f'  • SpanNER 160 Accuracy     : {sp_corr}/{n_total} ({sp_corr / n_total * 100:.2f}%)')
    print(f'  • Prompt v1 (With Label)   : {v1_corr}/{n_total} ({v1_corr / n_total * 100:.2f}%)')
    print(f'  • Prompt v2 (No Label)     : {v2_corr}/{n_total} ({v2_corr / n_total * 100:.2f}%)')
    print(f'  • Oracle Upper Bound       : {oracle_corr}/{n_total} ({oracle_corr / n_total * 100:.2f}%)')
    print(f'--------------------------------------------------------------------------------')
    print(f'  🔍 Prompt v1 Dynamics:')
    print(f'     - Rescued spans (SpanNER Wrong, v1 Correct) : +{v1_rescues} (+{v1_rescues / n_total * 100:.1f}%)')
    print(f'     - Degraded spans (SpanNER Correct, v1 Wrong): -{v1_degrades} (-{v1_degrades / n_total * 100:.1f}%)')
    print(f'     - Net Gain from Unconditional v1 Escalation : +{v1_rescues - v1_degrades} spans')
    print(f'  🔍 Prompt v2 Dynamics:')
    print(f'     - Rescued spans (SpanNER Wrong, v2 Correct) : +{v2_rescues} (+{v2_rescues / n_total * 100:.1f}%)')
    print(f'     - Degraded spans (SpanNER Correct, v2 Wrong): -{v2_degrades} (-{v2_degrades / n_total * 100:.1f}%)')
    print(f'     - Net Gain from Unconditional v2 Escalation : +{v2_rescues - v2_degrades} spans')
    print(f'  🔍 Concordance:')
    print(f'     - Agreement (v1 == v2) : {concordance}/{n_total} ({concordance / n_total * 100:.1f}%)')
    print('=' * 96)


In [ ]:
# ==============================================================================
# [CELL 11]: Export Unified Datasets inside 'Gating NN/'
# ==============================================================================
combined_csv = os.path.join(GATING_NN_DIR, 'gating_dataset_400train_dual_prompts.csv')
combined_jsonl = os.path.join(GATING_NN_DIR, 'gating_dataset_400train_dual_prompts.jsonl')

# 1. Save Full Combined Dataset
df_all.to_csv(combined_csv, index=False)
with open(combined_jsonl, 'w', encoding='utf-8') as f:
    for rec in df_all.to_dict(orient='records'):
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

# 2. Save Separate v1 and v2 Slices
cols_v1 = [
    'doc_id', 'span_text', 'start_char', 'end_char', 'gt_label',
    'spanner_label', 'spanner_confidence', 'spanner_top2_prob', 'second_best_label',
    'margin', 'uncertainty', 'uncertainty_method', 'novelty_score', 'p_background_o',
    'llm_v1_label', 'llm_v1_predicted_tag', 'llm_v1_rationale', 'llm_v1_correct',
    'v1_outcome', 'spanner_correct', 'v1_escalation_benefit'
]
df_v1 = df_all[[c for c in cols_v1 if c in df_all.columns]].rename(columns={
    'llm_v1_label': 'llm_label',
    'llm_v1_predicted_tag': 'llm_predicted_tag',
    'llm_v1_rationale': 'llm_rationale',
    'llm_v1_correct': 'llm_correct',
    'v1_escalation_benefit': 'escalation_benefit',
    'v1_outcome': 'outcome'
})
df_v1.to_csv(os.path.join(GATING_NN_DIR, 'gating_dataset_400train_v1.csv'), index=False)
with open(os.path.join(GATING_NN_DIR, 'gating_dataset_400train_v1.jsonl'), 'w', encoding='utf-8') as f:
    for rec in df_v1.to_dict(orient='records'):
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

cols_v2 = [
    'doc_id', 'span_text', 'start_char', 'end_char', 'gt_label',
    'spanner_label', 'spanner_confidence', 'spanner_top2_prob', 'second_best_label',
    'margin', 'uncertainty', 'uncertainty_method', 'novelty_score', 'p_background_o',
    'llm_v2_label', 'llm_v2_rationale', 'llm_v2_correct',
    'v2_outcome', 'spanner_correct', 'v2_escalation_benefit'
]
df_v2 = df_all[[c for c in cols_v2 if c in df_all.columns]].rename(columns={
    'llm_v2_label': 'llm_label',
    'llm_v2_rationale': 'llm_rationale',
    'llm_v2_correct': 'llm_correct',
    'v2_escalation_benefit': 'escalation_benefit',
    'v2_outcome': 'outcome'
})
df_v2.to_csv(os.path.join(GATING_NN_DIR, 'gating_dataset_400train_v2.csv'), index=False)
with open(os.path.join(GATING_NN_DIR, 'gating_dataset_400train_v2.jsonl'), 'w', encoding='utf-8') as f:
    for rec in df_v2.to_dict(orient='records'):
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print('=' * 88)
print('💾 All Datasets & Reports Successfully Saved to:', GATING_NN_DIR)
print(f'   ├── Per-Abstract JSONs : {GATING_JSON_DIR}/ ({len(selected_files)} files)')
print(f'   ├── Per-Abstract Reports: {GATING_REPORTS_DIR}/ ({len(selected_files)} files)')
print(f'   ├── Dual Dataset CSV   : gating_dataset_400train_dual_prompts.csv ({len(df_all)} rows)')
print(f'   ├── Dual Dataset JSONL : gating_dataset_400train_dual_prompts.jsonl ({len(df_all)} rows)')
print(f'   ├── Prompt v1 Slice    : gating_dataset_400train_v1.csv ({len(df_v1)} rows)')
print(f'   └── Prompt v2 Slice    : gating_dataset_400train_v2.csv ({len(df_v2)} rows)')
print('=' * 88)
df_all.head(5)
